# database construction and data cleaning

In [5]:
import duckdb
import pandas as pd
import time

print("正在连接/创建本地 DuckDB 数据库...")
con = duckdb.connect('taobao_project.db')

print("开始执行 Phase 1: 数据清洗与底层宽表构建" )
start_time = time.time()

# 读取、解析时间戳、多维特征提取、过滤边界
con.execute("""
    CREATE TABLE IF NOT EXISTS user_behavior_cleaned AS
    WITH raw_data AS (
        SELECT * FROM read_csv('UserBehavior.csv', 
            header=False,
            columns={
                'user_id': 'BIGINT',
                'item_id': 'BIGINT',
                'category_id': 'BIGINT',
                'behavior_type': 'VARCHAR',
                'timestamp': 'BIGINT'
            }
        )
    )
    SELECT 
        user_id,
        item_id,
        category_id,
        behavior_type,
        to_timestamp(timestamp) AS action_time,
        
        CAST(to_timestamp(timestamp) AS DATE) AS action_date,
        EXTRACT(HOUR FROM to_timestamp(timestamp)) AS action_hour,
        DAYOFWEEK(to_timestamp(timestamp)) AS action_weekday 
        
    FROM raw_data
    WHERE 
        -- 【异常清洗】：剔除双十一/双十二蓄水期 (11.25-12.03) 之外的脏数据
        CAST(to_timestamp(timestamp) AS DATE) BETWEEN '2017-11-25' AND '2017-12-03';
""")

end_time = time.time()
print(f"✅ 数据清洗完毕！耗时: {round(end_time - start_time, 2)} 秒")

# 结果验证：查看有效数据情况
print("\n--- 清洗后的底层大盘概况 ---")
summary_df = con.execute("""
    SELECT 
        COUNT(*) AS total_valid_rows, 
        COUNT(DISTINCT user_id) AS total_users,
        MIN(action_date) AS min_date, 
        MAX(action_date) AS max_date 
    FROM user_behavior_cleaned
""").df()

print(summary_df)

con.close()

正在连接/创建本地 DuckDB 数据库...
开始执行 Phase 1: 数据清洗与底层宽表构建
✅ 数据清洗完毕！耗时: 0.07 秒

--- 清洗后的底层大盘概况 ---
   total_valid_rows  total_users   min_date   max_date
0         100095231       987991 2017-11-25 2017-12-03


# Macro Metrics & Funnel Construction

In [6]:
con = duckdb.connect('taobao_project.db')

print("Phase 2: 计算宏观大盘指标与转化漏斗...")

# 计算基础事件数与转化率
# 使用 WITH 语句（CTE）先算出各行为的总量，再在外层计算转化率百分比
funnel_query = """
    WITH behavior_counts AS (
        SELECT 
            COUNT(DISTINCT user_id) AS total_uv,
            SUM(CASE WHEN behavior_type = 'pv' THEN 1 ELSE 0 END) AS total_pv,
            SUM(CASE WHEN behavior_type = 'fav' THEN 1 ELSE 0 END) AS total_fav,
            SUM(CASE WHEN behavior_type = 'cart' THEN 1 ELSE 0 END) AS total_cart,
            SUM(CASE WHEN behavior_type = 'buy' THEN 1 ELSE 0 END) AS total_buy
        FROM user_behavior_cleaned
    )
    SELECT 
        total_uv AS "总独立访客(UV)",
        total_pv AS "总浏览量(PV)",
        total_cart AS "加购总数(Cart)",
        total_fav AS "收藏总数(Fav)",
        total_buy AS "购买总数(Buy)",
        
    
        ROUND(total_cart * 100.0 / total_pv, 2) AS "浏览->加购 转化率(%)",
        ROUND(total_fav * 100.0 / total_pv, 2)  AS "浏览->收藏 转化率(%)",
        ROUND(total_buy * 100.0 / total_pv, 2)  AS "浏览->购买 总体转化率(%)"
    FROM behavior_counts;
"""

# 将 SQL 查询结果直接转换为 Pandas DataFrame 以便展示
funnel_df = con.execute(funnel_query).df()

print("\n--- 核心业务漏斗指标 ---")
print(funnel_df.T)

con.close()

Phase 2: 计算宏观大盘指标与转化漏斗...

--- 核心业务漏斗指标 ---
                           0
总独立访客(UV)          987991.00
总浏览量(PV)         89660688.00
加购总数(Cart)        5530446.00
收藏总数(Fav)         2888258.00
购买总数(Buy)         2015839.00
浏览->加购 转化率(%)           6.17
浏览->收藏 转化率(%)           3.22
浏览->购买 总体转化率(%)         2.25


# Multi-dimensional Attribution

In [7]:
con = duckdb.connect('taobao_project.db')

print("开始执行 Phase 3.1: 用户行为的小时级时序拆解 (Time-Series Attribution)...")

# 按小时 (action_hour) 分组统计
hourly_query = """
    SELECT 
        action_hour AS "小时",
        SUM(CASE WHEN behavior_type = 'pv' THEN 1 ELSE 0 END) AS "浏览量(PV)",
        SUM(CASE WHEN behavior_type = 'cart' THEN 1 ELSE 0 END) AS "加购量(Cart)",
        SUM(CASE WHEN behavior_type = 'buy' THEN 1 ELSE 0 END) AS "购买量(Buy)",
        ROUND(SUM(CASE WHEN behavior_type = 'buy' THEN 1 ELSE 0 END) * 100.0 / 
              SUM(CASE WHEN behavior_type = 'pv' THEN 1 ELSE 0 END), 2) AS "购买转化率(%)"
    FROM user_behavior_cleaned
    GROUP BY action_hour
    ORDER BY action_hour;
"""

# 提取为 DataFrame
hourly_df = con.execute(hourly_query).df()


print("\n--- 24小时用户行为趋势 ---")
print(hourly_df.to_string(index=False))

con.close()

开始执行 Phase 3.1: 用户行为的小时级时序拆解 (Time-Series Attribution)...

--- 24小时用户行为趋势 ---
 小时   浏览量(PV)  加购量(Cart)  购买量(Buy)  购买转化率(%)
  0 3056936.0   181807.0   57776.0      1.89
  1 1421673.0    85412.0   23169.0      1.63
  2  769269.0    46155.0   12012.0      1.56
  3  524992.0    32552.0    8026.0      1.53
  4  449628.0    28537.0    6748.0      1.50
  5  581694.0    37264.0    8135.0      1.40
  6 1226653.0    81305.0   18014.0      1.47
  7 2228959.0   147046.0   37679.0      1.69
  8 3042342.0   192036.0   64917.0      2.13
  9 3728498.0   229890.0   96134.0      2.58
 10 4334810.0   266963.0  127933.0      2.95
 11 4213518.0   260831.0  122048.0      2.90
 12 4255794.0   255811.0  118591.0      2.79
 13 4653934.0   279829.0  123427.0      2.65
 14 4642056.0   277093.0  122172.0      2.63
 15 4806706.0   284269.0  122733.0      2.55
 16 4607745.0   279035.0  116446.0      2.53
 17 4203397.0   255342.0  101301.0      2.41
 18 4313516.0   253193.0   95907.0      2.22
 19 5430878.0   314774

# RFE user segmentation model

In [8]:
con = duckdb.connect('taobao_project.db')

print("开始执行 Phase 4: 构建 RFE 用户价值分层模型...")

# 计算每个用户的 R, F, E 指标
rfe_query = """
    -- 建一张临时表来存储 RFE 结果，方便后续查询
    CREATE TABLE IF NOT EXISTS user_rfe_score AS
    SELECT 
        user_id,
        -- R: 距今最近一次活跃的天数 (假设当前观测日是 2017-12-04)
        CAST('2017-12-04' AS DATE) - MAX(action_date) AS recency_days,
        
        -- F: 活跃总天数
        COUNT(DISTINCT action_date) AS frequency_days,
        
        -- E: 互动深度加权得分
        SUM(
            CASE 
                WHEN behavior_type = 'pv' THEN 1
                WHEN behavior_type = 'fav' THEN 2
                WHEN behavior_type = 'cart' THEN 3
                WHEN behavior_type = 'buy' THEN 5
                ELSE 0 
            END
        ) AS engagement_score
    FROM user_behavior_cleaned
    GROUP BY user_id;
"""


con.execute(rfe_query)

# 跑出结果后，查看“互动得分最高”的 10 个超级死忠粉
top_users_query = """
    SELECT * FROM user_rfe_score 
    ORDER BY engagement_score DESC 
    LIMIT 10;
"""

top_users_df = con.execute(top_users_query).df()

print("\n--- 全站互动价值最高的 Top 10 用户 (超级 VIP) ---")
print(top_users_df)

con.close()

开始执行 Phase 4: 构建 RFE 用户价值分层模型...

--- 全站互动价值最高的 Top 10 用户 (超级 VIP) ---
   user_id  recency_days  frequency_days  engagement_score
0   486458             1               9            1441.0
1   162226             1               9            1097.0
2   500355             1               9            1092.0
3   866670             1               8            1050.0
4   922299             1               9             986.0
5   208947             1               9             980.0
6   326091             1               9             975.0
7   331457             1               8             975.0
8   655404             1               9             975.0
9   167945             1               9             936.0


In [9]:
funnel_df.T.to_csv('1_funnel_result.csv') 
hourly_df.to_csv('2_hourly_trend.csv', index=False)
top_users_df.to_csv('3_rfe_users.csv', index=False) 

print("数据已导出完毕，准备导入 Tableau！")

数据已导出完毕，准备导入 Tableau！
